<a href="https://colab.research.google.com/github/statisticnadi-maker/survival-analysis-oncology/blob/main/md_to_docx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U pypandoc_binary python-docx lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.9 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

import os
import re
import tempfile
import unicodedata
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import pypandoc
from docx import Document
from docx.enum.style import WD_STYLE_TYPE
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Pt
from lxml import etree


# ============================================================
# ثابت‌ها
# ============================================================

BidiMarkMode = Literal["NONE", "LRM", "LRI_PDI"]

LRM = "\u200e"
LRI = "\u2066"
PDI = "\u2069"

BIDI_CONTROL_CHARS = (
    "\u200e\u200f"
    "\u202a\u202b\u202c"
    "\u2066\u2067\u2068\u2069"
)

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
M_NS = "http://schemas.openxmlformats.org/officeDocument/2006/math"

NS = {
    "w": W_NS,
    "m": M_NS,
}


# ============================================================
# تنظیمات
# ============================================================

@dataclass
class ConverterConfig:
    input_dir: Path
    output_dir: Path

    reference_doc: Path | None = None

    persian_font: str = "B Nazanin"
    english_font: str = "Times New Roman"
    base_font_size: float = 12

    # NONE یا LRM یا LRI_PDI
    bidi_mark_mode: BidiMarkMode = "LRM"

    # متن عادی فارسی Justify شود؟
    justify_persian_text: bool = True

    # اندازه فونت تمام Runها اجباری تنظیم شود؟
    force_font_size: bool = False

    # اگر خروجی وجود داشت، بازنویسی شود؟
    overwrite: bool = False

    # پسوند فایل خروجی
    output_suffix: str = "_rtl"

    # پردازش Header و Footer
    process_headers_footers: bool = True

    # پردازش Footnote، Endnote و Comment
    process_notes_comments: bool = True

    # زبان‌های Word
    rtl_language: str = "fa-IR"
    ltr_language: str = "en-US"

    # تنظیمات ورودی Pandoc
    pandoc_format: str = (
        "markdown"
        "+tex_math_dollars"
        "+tex_math_single_backslash"
        "+raw_tex"
        "+pipe_tables"
        "+fenced_code_blocks"
        "+yaml_metadata_block"
        "+footnotes"
        "+definition_lists"
    )

    def __post_init__(self):
        self.input_dir = Path(self.input_dir)
        self.output_dir = Path(self.output_dir)

        if self.reference_doc:
            self.reference_doc = Path(self.reference_doc)

        self.bidi_mark_mode = self.bidi_mark_mode.upper()

        if self.bidi_mark_mode not in {"NONE", "LRM", "LRI_PDI"}:
            raise ValueError(
                "bidi_mark_mode باید NONE یا LRM یا LRI_PDI باشد."
            )


# ============================================================
# اتصال به Google Drive
# ============================================================

def mount_google_drive(
    mount_point: str = "/content/drive",
    force_remount: bool = False,
):
    """اتصال Google Drive در محیط Colab."""

    try:
        from google.colab import drive
    except ImportError:
        print("این محیط Google Colab نیست.")
        return False

    drive.mount(
        mount_point,
        force_remount=force_remount,
    )

    return True


def check_dependencies():
    """بررسی وجود Pandoc."""

    try:
        version = pypandoc.get_pandoc_version()
        print(f"Pandoc version: {version}")
        return str(version)

    except Exception as exc:
        raise RuntimeError(
            "Pandoc نصب نشده است. سلول نصب کتابخانه‌ها را اجرا کنید."
        ) from exc


# ============================================================
# مدیریت مسیر خروجی
# ============================================================

def ensure_directories(config: ConverterConfig):
    config.input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    config.output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


def get_unique_output_path(
    output_path: Path,
    overwrite: bool,
) -> Path:
    """جلوگیری از بازنویسی ناخواسته فایل."""

    if overwrite or not output_path.exists():
        return output_path

    counter = 2

    while True:
        candidate = output_path.with_name(
            f"{output_path.stem}_{counter}{output_path.suffix}"
        )

        if not candidate.exists():
            return candidate

        counter += 1


# ============================================================
# حفاظت از فرمول‌ها و ساختار Markdown
# ============================================================

class ProtectedMarkdown:
    """
    بخش‌های حساس Markdown را موقتاً با Placeholder جایگزین می‌کند.

    Placeholderها فقط شامل اعداد و کاراکترهای Private Use هستند
    تا الگوریتم BiDi آن‌ها را تغییر ندهد.
    """

    def __init__(
        self,
        start_marker: str = "\ue000",
        end_marker: str = "\ue001",
    ):
        self.items: list[str] = []

        self.start_marker = start_marker
        self.end_marker = end_marker

        self.placeholder_regex = re.compile(
            re.escape(start_marker)
            + r"(\d{8})"
            + re.escape(end_marker)
        )

    def store(self, value: str) -> str:
        index = len(self.items)
        self.items.append(value)

        return (
            f"{self.start_marker}"
            f"{index:08d}"
            f"{self.end_marker}"
        )

    def protect(
        self,
        text: str,
        pattern,
        flags: int = 0,
    ) -> str:

        if isinstance(pattern, str):
            pattern = re.compile(pattern, flags)

        return pattern.sub(
            lambda match: self.store(match.group(0)),
            text,
        )

    def restore(self, text: str) -> str:
        """بازگرداندن نواحی محافظت‌شده."""

        def replacement(match):
            index = int(match.group(1))
            return self.items[index]

        previous = None

        # برای Placeholderهای تو در تو
        while (
            previous != text
            and self.placeholder_regex.search(text)
        ):
            previous = text

            text = self.placeholder_regex.sub(
                replacement,
                text,
            )

        return text


def protect_markdown_regions(
    markdown_text: str,
):
    """
    حفاظت از:
    - YAML
    - فرمول‌ها
    - Code block
    - Inline code
    - لینک‌ها
    - URL
    - ایمیل
    - DOI
    - HTML
    """

    protected = ProtectedMarkdown()
    text = markdown_text

    # YAML Front Matter
    text = protected.protect(
        text,
        r"\A(?:\ufeff)?---[ \t]*\r?\n"
        r".*?"
        r"\r?\n(?:---|\.\.\.)[ \t]*"
        r"(?:\r?\n|$)",
        flags=re.DOTALL,
    )

    # Fenced code blocks
    text = protected.protect(
        text,
        r"^[ \t]{0,3}(`{3,}|~{3,})[^\n]*\n"
        r".*?"
        r"^[ \t]{0,3}\1[ \t]*$",
        flags=re.MULTILINE | re.DOTALL,
    )

    # HTML comments
    text = protected.protect(
        text,
        r"<!--.*?-->",
        flags=re.DOTALL,
    )

    # Display math: $$ ... $$
    text = protected.protect(
        text,
        r"(?<!\\)\$\$.*?(?<!\\)\$\$",
        flags=re.DOTALL,
    )

    # Display math: \[ ... \]
    text = protected.protect(
        text,
        r"\\\[.*?\\\]",
        flags=re.DOTALL,
    )

    # Inline math: \( ... \)
    text = protected.protect(
        text,
        r"\\\(.*?\\\)",
        flags=re.DOTALL,
    )

    # Inline code
    text = protected.protect(
        text,
        r"(?<!`)(`+)(?!`)"
        r"(.+?)"
        r"(?<!`)\1(?!`)",
        flags=re.DOTALL,
    )

    # Inline math: $ ... $
    text = protected.protect(
        text,
        r"(?<!\\)\$(?!\$)"
        r"(?:\\.|[^$\n])+?"
        r"(?<!\\)\$",
    )

    # Markdown links and images
    text = protected.protect(
        text,
        r"!?\[[^\]\n]*"
        r"(?:\\.[^\]\n]*)*"
        r"\]"
        r"\("
        r"(?:[^()\\\n]|\\.|\([^()\n]*\))*"
        r"\)",
    )

    # Reference-style links
    text = protected.protect(
        text,
        r"!?\[[^\]\n]+\]\[[^\]\n]*\]",
    )

    # Reference definitions
    text = protected.protect(
        text,
        r"^[ \t]{0,3}"
        r"\[[^\]\n]+\]:"
        r"[ \t]+\S+.*$",
        flags=re.MULTILINE,
    )

    # Pandoc citations
    text = protected.protect(
        text,
        r"\[(?:-?@[\w:.+-]+[^\]]*)\]",
    )

    # HTML autolink
    text = protected.protect(
        text,
        r"<(?:https?://|mailto:)[^>\n]+>",
    )

    # URL
    text = protected.protect(
        text,
        r"(?i)\b(?:https?://|www\.)"
        r"[^\s<>{}\[\]]+",
    )

    # Email
    text = protected.protect(
        text,
        r"(?i)\b"
        r"[A-Z0-9._%+-]+"
        r"@"
        r"[A-Z0-9.-]+"
        r"\.[A-Z]{2,}"
        r"\b",
    )

    # DOI
    text = protected.protect(
        text,
        r"(?i)\b"
        r"10\.\d{4,9}/"
        r"[-._;()/:A-Z0-9]+"
        r"\b",
    )

    # HTML tags
    text = protected.protect(
        text,
        r"</?[A-Za-z][^>\n]*>",
    )

    return text, protected


# ============================================================
# اصلاح BiDi در متن Markdown
# ============================================================

def wrap_ltr_unit(
    value: str,
    mode: BidiMarkMode,
) -> str:

    if not value or mode == "NONE":
        return value

    if mode == "LRM":
        return f"{LRM}{value}{LRM}"

    if mode == "LRI_PDI":
        return f"{LRI}{value}{PDI}"

    return value


# عبارات داخل پرانتز که حروف انگلیسی دارند
PAREN_LATIN_PATTERNS = [
    re.compile(
        r"\("
        r"(?=[^()\n]{1,220}\))"
        r"(?=[^()\n]*[A-Za-z])"
        r"[^()\n]+"
        r"\)"
    ),
    re.compile(
        r"\["
        r"(?=[^\[\]\n]{1,220}\])"
        r"(?=[^\[\]\n]*[A-Za-z])"
        r"[^\[\]\n]+"
        r"\]"
    ),
]


# عبارت‌های انگلیسی مانند:
# 95% CI
# p-value
# R version 4.4
# OR = 1.25
LATIN_CANDIDATE_REGEX = re.compile(
    r"(?<![\w\u0600-\u06ff])"
    r"("
    r"(?:"
    r"[A-Za-z0-9]+"
    r"(?:[._/+&'’%\-–—:][A-Za-z0-9]+)*"
    r"%?"
    r"|"
    r"[<>=≤≥±]+"
    r")"
    r"(?:"
    r"[ \t]+"
    r"(?:"
    r"[A-Za-z0-9]+"
    r"(?:[._/+&'’%\-–—:][A-Za-z0-9]+)*"
    r"%?"
    r"|"
    r"[<>=≤≥±]+"
    r")"
    r"){0,20}"
    r")"
    r"(?![\w\u0600-\u06ff])"
)


def fix_bidi_in_plain_text(
    text: str,
    mode: BidiMarkMode = "LRM",
) -> str:
    """
    اصلاح جهت عبارات انگلیسی خارج از فرمول و Code.
    """

    mode = mode.upper()

    if mode == "NONE":
        return text

    temporary = ProtectedMarkdown(
        start_marker="\ue002",
        end_marker="\ue003",
    )

    # ابتدا عبارت‌های کامل داخل پرانتز محافظت می‌شوند
    for pattern in PAREN_LATIN_PATTERNS:
        text = pattern.sub(
            lambda match: temporary.store(
                wrap_ltr_unit(
                    match.group(0),
                    mode,
                )
            ),
            text,
        )

    def replace_candidate(match):
        candidate = match.group(1)

        if not re.search(r"[A-Za-z]", candidate):
            return candidate

        return wrap_ltr_unit(
            candidate,
            mode,
        )

    text = LATIN_CANDIDATE_REGEX.sub(
        replace_candidate,
        text,
    )

    return temporary.restore(text)


def preprocess_markdown(
    markdown_text: str,
    config: ConverterConfig,
) -> str:
    """
    حفاظت از فرمول‌ها، اعمال BiDi و بازگرداندن فرمول‌ها.
    """

    protected_text, protected = protect_markdown_regions(
        markdown_text
    )

    fixed_text = fix_bidi_in_plain_text(
        protected_text,
        config.bidi_mark_mode,
    )

    return protected.restore(fixed_text)


# ============================================================
# تشخیص جهت متن
# ============================================================

def is_rtl_letter(char: str) -> bool:
    code = ord(char)

    in_rtl_block = (
        0x0600 <= code <= 0x06FF
        or 0x0750 <= code <= 0x077F
        or 0x08A0 <= code <= 0x08FF
        or 0xFB50 <= code <= 0xFDFF
        or 0xFE70 <= code <= 0xFEFF
    )

    return (
        in_rtl_block
        and unicodedata.category(char).startswith("L")
    )


def is_latin_letter(char: str) -> bool:
    if not unicodedata.category(char).startswith("L"):
        return False

    return "LATIN" in unicodedata.name(char, "")


def classify_direction(text: str) -> str:
    """
    خروجی:
    rtl
    ltr
    mixed_rtl
    mixed_ltr
    neutral
    """

    translation_table = {
        ord(char): None
        for char in BIDI_CONTROL_CHARS
    }

    clean_text = text.translate(translation_table)

    rtl_count = sum(
        1 for char in clean_text
        if is_rtl_letter(char)
    )

    latin_count = sum(
        1 for char in clean_text
        if is_latin_letter(char)
    )

    if rtl_count == 0 and latin_count == 0:
        return "neutral"

    if rtl_count > 0 and latin_count == 0:
        return "rtl"

    if latin_count > 0 and rtl_count == 0:
        return "ltr"

    if rtl_count >= latin_count:
        return "mixed_rtl"

    return "mixed_ltr"


# ============================================================
# ابزارهای XML مربوط به Word
# ============================================================

def get_or_add_child(parent, tag: str):
    child = parent.find(qn(tag))

    if child is None:
        child = OxmlElement(tag)
        parent.append(child)

    return child


def set_on_off_property(
    parent,
    tag: str,
    enabled: bool,
):
    element = get_or_add_child(
        parent,
        tag,
    )

    element.set(
        qn("w:val"),
        "1" if enabled else "0",
    )


def paragraph_has_math(paragraph) -> bool:
    """بررسی وجود فرمول OMML در پاراگراف."""

    return bool(
        paragraph._p.xpath(
            ".//m:oMath | .//m:oMathPara"
        )
    )


def run_has_math(run) -> bool:
    return bool(
        run._r.xpath(
            ".//m:oMath | .//m:oMathPara"
        )
    )


def set_paragraph_rtl(
    paragraph,
    enabled: bool,
):
    """
    RTL واقعی پاراگراف با w:bidi.
    """

    paragraph_properties = (
        paragraph._p.get_or_add_pPr()
    )

    set_on_off_property(
        paragraph_properties,
        "w:bidi",
        enabled,
    )


def set_run_rtl(
    run,
    enabled: bool,
):
    """
    تنظیم جهت Run با w:rtl.
    """

    run_properties = (
        run._r.get_or_add_rPr()
    )

    set_on_off_property(
        run_properties,
        "w:rtl",
        enabled,
    )


def set_table_rtl(
    table,
    enabled: bool,
):
    """
    تنظیم جهت دیداری جدول با w:bidiVisual.
    """

    table_properties = table._tbl.tblPr

    bidi_visual = get_or_add_child(
        table_properties,
        "w:bidiVisual",
    )

    bidi_visual.set(
        qn("w:val"),
        "1" if enabled else "0",
    )


# ============================================================
# تنظیم فونت
# ============================================================

def set_run_fonts(
    run,
    config: ConverterConfig,
    rtl_run: bool,
):
    """
    فونت فارسی در w:cs و فونت انگلیسی در ascii/hAnsi.
    """

    run_properties = (
        run._r.get_or_add_rPr()
    )

    fonts = get_or_add_child(
        run_properties,
        "w:rFonts",
    )

    fonts.set(
        qn("w:ascii"),
        config.english_font,
    )

    fonts.set(
        qn("w:hAnsi"),
        config.english_font,
    )

    fonts.set(
        qn("w:eastAsia"),
        config.english_font,
    )

    fonts.set(
        qn("w:cs"),
        config.persian_font,
    )

    language = get_or_add_child(
        run_properties,
        "w:lang",
    )

    language.set(
        qn("w:val"),
        config.ltr_language,
    )

    language.set(
        qn("w:eastAsia"),
        config.ltr_language,
    )

    language.set(
        qn("w:bidi"),
        config.rtl_language,
    )

    if config.force_font_size:
        half_points = str(
            int(round(config.base_font_size * 2))
        )

        size = get_or_add_child(
            run_properties,
            "w:sz",
        )

        size.set(
            qn("w:val"),
            half_points,
        )

        size_complex = get_or_add_child(
            run_properties,
            "w:szCs",
        )

        size_complex.set(
            qn("w:val"),
            half_points,
        )

    set_run_rtl(
        run,
        rtl_run,
    )


def set_style_fonts(
    document,
    config: ConverterConfig,
):
    """
    تنظیم فونت Styleهای Word.
    اندازه Headingها تغییر داده نمی‌شود.
    """

    for style in document.styles:

        if style.type not in {
            WD_STYLE_TYPE.PARAGRAPH,
            WD_STYLE_TYPE.CHARACTER,
        }:
            continue

        try:
            style.font.name = config.english_font

            run_properties = (
                style.element.get_or_add_rPr()
            )

            fonts = get_or_add_child(
                run_properties,
                "w:rFonts",
            )

            fonts.set(
                qn("w:ascii"),
                config.english_font,
            )

            fonts.set(
                qn("w:hAnsi"),
                config.english_font,
            )

            fonts.set(
                qn("w:eastAsia"),
                config.english_font,
            )

            fonts.set(
                qn("w:cs"),
                config.persian_font,
            )

            language = get_or_add_child(
                run_properties,
                "w:lang",
            )

            language.set(
                qn("w:val"),
                config.ltr_language,
            )

            language.set(
                qn("w:bidi"),
                config.rtl_language,
            )

            if style.name == "Normal":
                style.font.size = Pt(
                    config.base_font_size
                )

                size_complex = get_or_add_child(
                    run_properties,
                    "w:szCs",
                )

                size_complex.set(
                    qn("w:val"),
                    str(
                        int(
                            config.base_font_size * 2
                        )
                    ),
                )

        except Exception:
            continue


# ============================================================
# پردازش پاراگراف و جدول
# ============================================================

def get_style_name(paragraph) -> str:
    try:
        return (
            paragraph.style.name or ""
        ).lower()

    except Exception:
        return ""


def is_code_paragraph(paragraph) -> bool:
    """
    تشخیص Code Block ایجادشده توسط Pandoc.
    """

    style_name = get_style_name(paragraph)

    if any(
        token in style_name
        for token in (
            "source code",
            "verbatim",
            "code block",
        )
    ):
        return True

    non_empty_runs = [
        run
        for run in paragraph.runs
        if run.text.strip()
    ]

    if not non_empty_runs:
        return False

    return all(
        run.style is not None
        and any(
            token in (
                run.style.name or ""
            ).lower()
            for token in (
                "verbatim",
                "source code",
                "code",
            )
        )
        for run in non_empty_runs
    )


def get_rtl_alignment(
    paragraph,
    config: ConverterConfig,
):
    """
    تیترها و Captionها راست‌چین؛ متن عادی Justify.
    """

    style_name = get_style_name(paragraph)

    special_styles = (
        "heading",
        "title",
        "subtitle",
        "caption",
        "list paragraph",
    )

    if any(
        token in style_name
        for token in special_styles
    ):
        return WD_ALIGN_PARAGRAPH.RIGHT

    if config.justify_persian_text:
        return WD_ALIGN_PARAGRAPH.JUSTIFY

    return WD_ALIGN_PARAGRAPH.RIGHT


def process_paragraph(
    paragraph,
    config: ConverterConfig,
    default_rtl: bool = True,
):
    """
    پردازش پاراگراف بدون استفاده از paragraph.text = ...
    در نتیجه ساختار فرمول و Formatting خراب نمی‌شود.
    """

    contains_math = paragraph_has_math(
        paragraph
    )

    text = paragraph.text or ""

    # فرمول نمایشی بدون متن معمولی
    if contains_math and not text.strip():

        paragraph.alignment = (
            WD_ALIGN_PARAGRAPH.CENTER
        )

        return

    # Code block باید LTR باشد
    if is_code_paragraph(paragraph):

        set_paragraph_rtl(
            paragraph,
            False,
        )

        paragraph.alignment = (
            WD_ALIGN_PARAGRAPH.LEFT
        )

        for run in paragraph.runs:

            if run_has_math(run):
                continue

            set_run_fonts(
                run,
                config,
                rtl_run=False,
            )

        return

    direction = classify_direction(text)

    paragraph_rtl = direction in {
        "rtl",
        "mixed_rtl",
    }

    if direction == "neutral":
        paragraph_rtl = default_rtl

    set_paragraph_rtl(
        paragraph,
        paragraph_rtl,
    )

    if paragraph_rtl:

        paragraph.alignment = get_rtl_alignment(
            paragraph,
            config,
        )

    else:

        paragraph.alignment = (
            WD_ALIGN_PARAGRAPH.LEFT
        )

    # پردازش Runها
    for run in paragraph.runs:

        # عناصر OMML نباید دست‌کاری متنی شوند
        if run_has_math(run):
            continue

        run_direction = classify_direction(
            run.text or ""
        )

        if run_direction in {
            "rtl",
            "mixed_rtl",
        }:
            rtl_run = True

        elif run_direction in {
            "ltr",
            "mixed_ltr",
        }:
            rtl_run = False

        else:
            rtl_run = paragraph_rtl

        set_run_fonts(
            run,
            config,
            rtl_run=rtl_run,
        )


def get_table_text(table) -> str:
    parts = []
    visited_cells = set()

    for row in table.rows:
        for cell in row.cells:

            cell_id = id(cell._tc)

            if cell_id in visited_cells:
                continue

            visited_cells.add(cell_id)

            for paragraph in cell.paragraphs:
                parts.append(paragraph.text)

    return " ".join(parts)


def process_table(
    table,
    config: ConverterConfig,
):
    """
    پردازش جدول و جدول‌های تو در تو.
    """

    table_direction = classify_direction(
        get_table_text(table)
    )

    table_rtl = table_direction in {
        "rtl",
        "mixed_rtl",
    }

    if table_direction == "neutral":
        table_rtl = True

    set_table_rtl(
        table,
        table_rtl,
    )

    visited_cells = set()

    for row in table.rows:
        for cell in row.cells:

            cell_id = id(cell._tc)

            if cell_id in visited_cells:
                continue

            visited_cells.add(cell_id)

            for paragraph in cell.paragraphs:
                process_paragraph(
                    paragraph,
                    config,
                    default_rtl=table_rtl,
                )

            for nested_table in cell.tables:
                process_table(
                    nested_table,
                    config,
                )


def process_container(
    container,
    config: ConverterConfig,
):
    """
    پردازش Body، Header یا Footer.
    """

    for paragraph in container.paragraphs:
        process_paragraph(
            paragraph,
            config,
            default_rtl=True,
        )

    for table in container.tables:
        process_table(
            table,
            config,
        )


# ============================================================
# پردازش Footnote، Endnote و Comment
# ============================================================

def xml_get_or_add(
    parent,
    namespace: str,
    local_name: str,
    first: bool = False,
):
    tag = f"{{{namespace}}}{local_name}"

    child = parent.find(tag)

    if child is None:
        child = etree.Element(tag)

        if first:
            parent.insert(0, child)
        else:
            parent.append(child)

    return child


def xml_set_value(
    element,
    value: str,
):
    element.set(
        f"{{{W_NS}}}val",
        value,
    )


def patch_xml_paragraph(
    paragraph_element,
    config: ConverterConfig,
):
    """
    اصلاح پاراگراف‌های Footnote، Endnote و Comment.
    """

    text = "".join(
        paragraph_element.xpath(
            ".//w:t/text()",
            namespaces=NS,
        )
    )

    contains_math = bool(
        paragraph_element.xpath(
            ".//m:oMath | .//m:oMathPara",
            namespaces=NS,
        )
    )

    if contains_math and not text.strip():

        paragraph_properties = xml_get_or_add(
            paragraph_element,
            W_NS,
            "pPr",
            first=True,
        )

        alignment = xml_get_or_add(
            paragraph_properties,
            W_NS,
            "jc",
        )

        xml_set_value(
            alignment,
            "center",
        )

        return

    direction = classify_direction(text)

    paragraph_rtl = direction in {
        "rtl",
        "mixed_rtl",
        "neutral",
    }

    paragraph_properties = xml_get_or_add(
        paragraph_element,
        W_NS,
        "pPr",
        first=True,
    )

    bidi = xml_get_or_add(
        paragraph_properties,
        W_NS,
        "bidi",
    )

    xml_set_value(
        bidi,
        "1" if paragraph_rtl else "0",
    )

    alignment = xml_get_or_add(
        paragraph_properties,
        W_NS,
        "jc",
    )

    xml_set_value(
        alignment,
        "right" if paragraph_rtl else "left",
    )

    runs = paragraph_element.xpath(
        "./w:r | .//w:hyperlink/w:r",
        namespaces=NS,
    )

    for run in runs:

        if run.xpath(
            ".//m:oMath | .//m:oMathPara",
            namespaces=NS,
        ):
            continue

        run_text = "".join(
            run.xpath(
                ".//w:t/text()",
                namespaces=NS,
            )
        )

        run_direction = classify_direction(
            run_text
        )

        if run_direction in {
            "rtl",
            "mixed_rtl",
        }:
            run_rtl = True

        elif run_direction in {
            "ltr",
            "mixed_ltr",
        }:
            run_rtl = False

        else:
            run_rtl = paragraph_rtl

        run_properties = xml_get_or_add(
            run,
            W_NS,
            "rPr",
            first=True,
        )

        rtl_property = xml_get_or_add(
            run_properties,
            W_NS,
            "rtl",
        )

        xml_set_value(
            rtl_property,
            "1" if run_rtl else "0",
        )

        fonts = xml_get_or_add(
            run_properties,
            W_NS,
            "rFonts",
        )

        fonts.set(
            f"{{{W_NS}}}ascii",
            config.english_font,
        )

        fonts.set(
            f"{{{W_NS}}}hAnsi",
            config.english_font,
        )

        fonts.set(
            f"{{{W_NS}}}eastAsia",
            config.english_font,
        )

        fonts.set(
            f"{{{W_NS}}}cs",
            config.persian_font,
        )


def patch_auxiliary_xml_parts(
    docx_path: Path,
    config: ConverterConfig,
):
    """
    پردازش XML پاورقی‌ها، Endnoteها و Commentها.
    """

    target_parts = {
        "word/footnotes.xml",
        "word/endnotes.xml",
        "word/comments.xml",
    }

    temporary_path = docx_path.with_suffix(
        ".temporary.docx"
    )

    changed = False

    with zipfile.ZipFile(
        docx_path,
        "r",
    ) as source_zip:

        with zipfile.ZipFile(
            temporary_path,
            "w",
        ) as target_zip:

            for item in source_zip.infolist():

                data = source_zip.read(
                    item.filename
                )

                if item.filename in target_parts:

                    root = etree.fromstring(data)

                    paragraphs = root.xpath(
                        ".//w:p",
                        namespaces=NS,
                    )

                    for paragraph in paragraphs:
                        patch_xml_paragraph(
                            paragraph,
                            config,
                        )

                    data = etree.tostring(
                        root,
                        xml_declaration=True,
                        encoding="UTF-8",
                        standalone=True,
                    )

                    changed = True

                target_zip.writestr(
                    item,
                    data,
                )

    if changed:
        os.replace(
            temporary_path,
            docx_path,
        )

    elif temporary_path.exists():
        temporary_path.unlink()


# ============================================================
# پس‌پردازش فایل Word
# ============================================================

def postprocess_docx(
    input_docx: Path,
    output_docx: Path,
    config: ConverterConfig,
):
    """
    اصلاح RTL و فونت پس از تبدیل Pandoc.
    """

    document = Document(
        str(input_docx)
    )

    set_style_fonts(
        document,
        config,
    )

    process_container(
        document,
        config,
    )

    # Header و Footer
    if config.process_headers_footers:

        processed_parts = set()

        header_footer_names = (
            "header",
            "first_page_header",
            "even_page_header",
            "footer",
            "first_page_footer",
            "even_page_footer",
        )

        for section in document.sections:

            for attribute_name in header_footer_names:

                part_proxy = getattr(
                    section,
                    attribute_name,
                )

                if part_proxy.is_linked_to_previous:
                    continue

                part_name = str(
                    part_proxy.part.partname
                )

                if part_name in processed_parts:
                    continue

                processed_parts.add(part_name)

                process_container(
                    part_proxy,
                    config,
                )

    output_docx.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    document.save(
        str(output_docx)
    )

    if config.process_notes_comments:

        patch_auxiliary_xml_parts(
            output_docx,
            config,
        )


# ============================================================
# تبدیل با Pandoc
# ============================================================

def build_pandoc_arguments(
    source_file: Path,
    config: ConverterConfig,
):
    arguments = [
        "--standalone",
        "--wrap=preserve",
        f"--metadata=lang:{config.rtl_language}",
    ]

    # تصاویر نسبی کنار فایل Markdown پیدا شوند
    arguments.append(
        f"--resource-path={source_file.parent.resolve()}"
    )

    # فایل Reference Word
    if config.reference_doc:

        if not config.reference_doc.exists():
            raise FileNotFoundError(
                f"Reference DOCX پیدا نشد: "
                f"{config.reference_doc}"
            )

        arguments.append(
            f"--reference-doc="
            f"{config.reference_doc.resolve()}"
        )

    return arguments


def pandoc_convert(
    markdown_file: Path,
    output_docx: Path,
    config: ConverterConfig,
):
    """
    تبدیل Markdown به DOCX.

    Pandoc فرمول‌های LaTeX را برای DOCX به OMML تبدیل می‌کند.
    از --mathml استفاده نمی‌شود.
    """

    arguments = build_pandoc_arguments(
        markdown_file,
        config,
    )

    pypandoc.convert_file(
        source_file=str(markdown_file),
        to="docx",
        format=config.pandoc_format,
        outputfile=str(output_docx),
        extra_args=arguments,
    )

    if (
        not output_docx.exists()
        or output_docx.stat().st_size == 0
    ):
        raise RuntimeError(
            "Pandoc فایل Word معتبر ایجاد نکرد."
        )


# ============================================================
# اعتبارسنجی خروجی
# ============================================================

def validate_docx(
    docx_path: Path,
):
    """
    بررسی:
    - ZIP معتبر
    - وجود document.xml
    - تعداد OMML
    - تعداد w:bidi
    - تعداد w:rtl
    - بازشدن با python-docx
    """

    result = {
        "valid": False,
        "file_size": 0,
        "document_xml": False,
        "omml_count": 0,
        "bidi_paragraph_count": 0,
        "rtl_run_count": 0,
        "opens_with_python_docx": False,
        "errors": [],
    }

    if not docx_path.exists():

        result["errors"].append(
            "فایل خروجی وجود ندارد."
        )

        return result

    result["file_size"] = (
        docx_path.stat().st_size
    )

    try:

        with zipfile.ZipFile(
            docx_path,
            "r",
        ) as archive:

            bad_file = archive.testzip()

            if bad_file:
                result["errors"].append(
                    f"عضو خراب در DOCX: {bad_file}"
                )

            names = set(
                archive.namelist()
            )

            result["document_xml"] = (
                "word/document.xml" in names
            )

            xml_files = [
                name
                for name in names
                if name.startswith("word/")
                and name.endswith(".xml")
            ]

            for xml_file in xml_files:

                try:
                    root = etree.fromstring(
                        archive.read(xml_file)
                    )

                except etree.XMLSyntaxError:
                    continue

                result["omml_count"] += len(
                    root.xpath(
                        ".//m:oMath | .//m:oMathPara",
                        namespaces=NS,
                    )
                )

                result["bidi_paragraph_count"] += len(
                    root.xpath(
                        ".//w:pPr/w:bidi",
                        namespaces=NS,
                    )
                )

                result["rtl_run_count"] += len(
                    root.xpath(
                        ".//w:rPr/w:rtl",
                        namespaces=NS,
                    )
                )

    except Exception as exc:

        result["errors"].append(
            f"خطا در ساختار DOCX: {exc}"
        )

    try:

        Document(
            str(docx_path)
        )

        result["opens_with_python_docx"] = True

    except Exception as exc:

        result["errors"].append(
            f"فایل با python-docx باز نشد: {exc}"
        )

    result["valid"] = all([
        result["file_size"] > 0,
        result["document_xml"],
        result["opens_with_python_docx"],
        len(result["errors"]) == 0,
    ])

    return result


# ============================================================
# تبدیل یک فایل
# ============================================================

def convert_single_file(
    source_file: str | Path,
    config: ConverterConfig,
    output_file: str | Path | None = None,
):
    """
    تبدیل کامل یک فایل Markdown.
    """

    source_file = Path(source_file)

    if not source_file.exists():
        raise FileNotFoundError(
            f"فایل پیدا نشد: {source_file}"
        )

    if source_file.suffix.lower() not in {
        ".md",
        ".markdown",
    }:
        raise ValueError(
            "فایل ورودی باید پسوند md یا markdown داشته باشد."
        )

    ensure_directories(config)

    if output_file:

        desired_output = Path(
            output_file
        )

    else:

        desired_output = (
            config.output_dir
            / (
                source_file.stem
                + config.output_suffix
                + ".docx"
            )
        )

    final_output = get_unique_output_path(
        desired_output,
        config.overwrite,
    )

    print(
        f"در حال خواندن فایل: "
        f"{source_file.name}"
    )

    raw_markdown = source_file.read_text(
        encoding="utf-8-sig"
    )

    processed_markdown = preprocess_markdown(
        raw_markdown,
        config,
    )

    with tempfile.TemporaryDirectory(
        prefix="markdown_to_word_"
    ) as temporary_directory:

        temporary_directory = Path(
            temporary_directory
        )

        temporary_markdown = (
            temporary_directory
            / "preprocessed.md"
        )

        temporary_docx = (
            temporary_directory
            / "pandoc_output.docx"
        )

        temporary_markdown.write_text(
            processed_markdown,
            encoding="utf-8",
        )

        print(
            "در حال تبدیل Markdown به Word..."
        )

        pandoc_convert(
            temporary_markdown,
            temporary_docx,
            config,
        )

        print(
            "در حال اصلاح RTL و فونت‌ها..."
        )

        postprocess_docx(
            temporary_docx,
            final_output,
            config,
        )

    validation = validate_docx(
        final_output
    )

    if not validation["valid"]:

        raise RuntimeError(
            "اعتبارسنجی خروجی ناموفق بود: "
            + " | ".join(
                validation["errors"]
            )
        )

    print("\nتبدیل با موفقیت انجام شد.")
    print(f"فایل خروجی: {final_output}")
    print(
        f"حجم فایل: "
        f"{validation['file_size']:,} bytes"
    )
    print(
        f"تعداد عناصر OMML: "
        f"{validation['omml_count']}"
    )
    print(
        f"تعداد پاراگراف‌های BiDi: "
        f"{validation['bidi_paragraph_count']}"
    )
    print(
        f"تعداد ویژگی‌های RTL در Runها: "
        f"{validation['rtl_run_count']}"
    )

    return final_output, validation


# ============================================================
# تبدیل دسته‌ای پوشه
# ============================================================

def convert_directory(
    config: ConverterConfig,
    recursive: bool = False,
):
    """
    تبدیل تمام فایل‌های md یک پوشه.
    """

    ensure_directories(config)

    search_pattern = (
        "**/*"
        if recursive
        else "*"
    )

    markdown_files = sorted([
        path
        for path in config.input_dir.glob(
            search_pattern
        )
        if path.is_file()
        and path.suffix.lower() in {
            ".md",
            ".markdown",
        }
    ])

    if not markdown_files:

        print(
            "هیچ فایل Markdown در مسیر زیر پیدا نشد:"
        )

        print(
            config.input_dir
        )

        return []

    reports = []

    for markdown_file in markdown_files:

        print("\n" + "=" * 70)

        try:

            output, validation = convert_single_file(
                markdown_file,
                config,
            )

            reports.append({
                "source": markdown_file,
                "output": output,
                "success": True,
                "validation": validation,
                "error": None,
            })

        except Exception as exc:

            print(
                f"خطا در تبدیل "
                f"{markdown_file.name}: {exc}"
            )

            reports.append({
                "source": markdown_file,
                "output": None,
                "success": False,
                "validation": None,
                "error": str(exc),
            })

    successful = sum(
        report["success"]
        for report in reports
    )

    print("\n" + "=" * 70)

    print(
        f"نتیجه نهایی: "
        f"{successful} فایل از "
        f"{len(reports)} فایل تبدیل شد."
    )

    return reports


# ============================================================
# ساخت فایل آزمایشی
# ============================================================

SAMPLE_MARKDOWN = r"""
# تحلیل رگرسیون لجستیک

در مدل نهایی، مقدار `p-value` برای متغیر سن برابر با $p = 0.032$ بود.

نسبت شانس تعدیل‌شده برابر با (OR = 1.25, 95% CI: 1.10–1.42) گزارش شد.

مدل مورد استفاده به‌صورت زیر است:

$$
\log\left(\frac{p}{1-p}\right)
=
\beta_0 + \beta_1X_1 + \beta_2X_2
$$

برای تحلیل از R version 4.4 و بسته `survival` استفاده شد.

## جدول نتایج

| متغیر | OR | 95% CI | p-value |
|---|---:|---:|---:|
| سن | 1.25 | 1.10–1.42 | 0.032 |
| جنسیت | 0.87 | 0.72–1.05 | 0.140 |

یک لینک آزمایشی:

[وب‌سایت Pandoc](https://pandoc.org)

یک پاورقی آزمایشی.[^1]

[^1]: این پاورقی شامل عبارت English term و فرمول $x^2$ است.
"""


def create_sample_file(
    output_path: str | Path,
):
    """
    ایجاد فایل Markdown آزمایشی.
    """

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path.write_text(
        SAMPLE_MARKDOWN.strip(),
        encoding="utf-8",
    )

    print(
        f"فایل آزمایشی ساخته شد: "
        f"{output_path}"
    )

    return output_path


print("کد مبدل با موفقیت بارگذاری شد.")

کد مبدل با موفقیت بارگذاری شد.


In [5]:
from pathlib import Path

config = ConverterConfig(
    input_dir=Path("/content"),
    output_dir=Path("/content"),

    reference_doc=None,

    persian_font="B Nazanin",
    english_font="Times New Roman",
    base_font_size=12,

    bidi_mark_mode="LRM",
    justify_persian_text=True,
    force_font_size=False,

    overwrite=False,
    output_suffix="_rtl",

    process_headers_footers=True,
    process_notes_comments=True,
)

ensure_directories(config)

print("پوشه ورودی:")
print(config.input_dir)

print("پوشه خروجی:")
print(config.output_dir)

پوشه ورودی:
/content
پوشه خروجی:
/content


In [8]:
source_file = Path("/content/test_bidi_math.md")

output_file, validation = convert_single_file(
    source_file,
    config,
)

print("فایل خروجی:")
print(output_file)

در حال خواندن فایل: test_bidi_math.md
در حال تبدیل Markdown به Word...
در حال اصلاح RTL و فونت‌ها...

تبدیل با موفقیت انجام شد.
فایل خروجی: /content/test_bidi_math_rtl.docx
حجم فایل: 15,733 bytes
تعداد عناصر OMML: 31
تعداد پاراگراف‌های BiDi: 73
تعداد ویژگی‌های RTL در Runها: 140
فایل خروجی:
/content/test_bidi_math_rtl.docx
